In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import simlib.generate_sims as gen_sims
import simlib.generate_timeseries as gen_times


In [ ]:
n_prop = 5
n_s = 15
pchange = np.linspace(-0.1, 0.1, n_s)
proportion_s0_r2s = np.linspace(0,1,n_prop)
decay_models = ["monoexponential", "taylor1_approx", "linear_approx"]
print(f"{pchange=}, {proportion_s0_r2s=}")

inputted_s = np.tile(pchange, (n_prop, 1))
s0_mean = 500
r2s_mean = 1/28
te_baseline = 28
delta_s0, delta_r2s = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
        inputted_s=inputted_s,
        s0_mean=s0_mean,
        r2s_mean=r2s_mean,
        proportion_s0_r2s=proportion_s0_r2s,
        te_baseline=te_baseline,
        prop_to_scale="signal"
    )


s = gen_sims.all_decay_models(
    tes=60, #te_baseline,
    mean_s0=s0_mean,
    mean_r2star=r2s_mean,
    delta_s0=delta_s0,
    delta_r2star=delta_r2s
)
#s =  gen_sims.monoexponential(tes=te_baseline, s0=s0_mean + delta_s0, r2star=r2s_mean + delta_r2s)
print(f"{s=}")


In [ ]:
# Sanity check that the delta_s0 and delta_r2s are doing what we expect
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
plt.plot(pchange,(delta_s0/s0_mean).T)
plt.subplot(1,3,2)
plt.plot(pchange,(-te_baseline*delta_r2s).T)
plt.subplot(1,3,3)
plt.plot(pchange,(delta_s0/s0_mean-te_baseline*delta_r2s).T)
# THIS IS WRONG BECAUSE the THIRD PANEL SHOULD ALL BE -0.1 to 0.1

plt.legend(proportion_s0_r2s)

In [ ]:
s0_mean_list = [10, 1000]
r2s_mean_list = [ 1/28, 1/40]
te_list = [20, 40, 60]


for s0_mean in s0_mean_list:
    for r2s_mean in r2s_mean_list:
        for te_write in te_list:
            delta_s0, delta_r2s = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
                    inputted_s=inputted_s,
                    s0_mean=s0_mean,
                    r2s_mean=r2s_mean,
                    proportion_s0_r2s=proportion_s0_r2s,
                    te_baseline=te_baseline,
                    prop_to_scale="signal"
            )


            s = gen_sims.all_decay_models(
                tes=te_write, #te_baseline,
                mean_s0=s0_mean,
                mean_r2star=r2s_mean,
                delta_s0=delta_s0,
                delta_r2star=delta_r2s
            )

            plt.figure(figsize=(15,10))
            plt.subplot(2, n_prop, 1)
            plt.ylabel("Raw Signal")
            for pltidx in range(n_prop):
                plt.subplot(2, n_prop, pltidx+1)
                plt.title(f"Proportion s0/r2s: {proportion_s0_r2s[pltidx]:.2f}\n{s0_mean=}\nr2s_mean={np.round(r2s_mean, decimals=3)}\nTE={te_write}")
                for midx, model in enumerate(decay_models):
                    plt.subplot(2, n_prop, pltidx+1)
                    plt.plot(100*pchange, s[model][pltidx,:], label=model)
                    if model == "monoexponential":
                        mono_exp_vals = s[model][pltidx,:]
                    else:
                        plt.subplot(2, n_prop, pltidx+1+n_prop)
                        plt.plot(100*pchange, 100*(s[model][pltidx,:] - mono_exp_vals)/mono_exp_vals, linestyle="dashed", label=model)
                        plt.ylabel("% change from monoexponential")
                
                plt.xlabel("Signal %change")
            plt.subplot(2, n_prop, n_prop) 
            plt.legend()
            plt.subplot(2, n_prop, 2*n_prop) 
            plt.legend()
            plt.show()
    

In [ ]:
s0_mean = 500
r2s_mean = 1/40
n_timepoints = 300
n_prop=6
proportion_s0_r2s = np.linspace(0,1,n_prop)
te_baseline = 28
tes = [10, 20, 30, 40, 50, 60]

sim_time = gen_times.gen_bandpass_randn_timeseries(n_reps=1, n_timepoints=n_timepoints, seed=0, passband = [1 / 100, 1 / 20], fs = 0.5,)
sim_time = np.squeeze(0.1 * sim_time / np.max(np.abs(sim_time)))

delta_s0_prop = np.full((n_timepoints, n_prop), np.nan)
delta_r2s_prop = np.full((n_timepoints, n_prop), np.nan)

sim_echoes = np.full((n_timepoints, len(tes), n_prop), np.nan)

for prop_idx, prop in enumerate(proportion_s0_r2s):
    delta_s0_prop[:, prop_idx], delta_r2s_prop[:, prop_idx] = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
                    inputted_s=sim_time,
                    s0_mean=s0_mean,
                    r2s_mean=r2s_mean,
                    proportion_s0_r2s=prop,
                    te_baseline=te_baseline,
                    prop_to_scale="signal"
            )


    for te_idx, te_write in enumerate(tes):
        sim_echoes[:, te_idx, prop_idx] = gen_sims.monoexponential(
                        tes=te_write, #te_baseline,
                        s0=s0_mean + delta_s0_prop[:, prop_idx],
                        r2star=r2s_mean + delta_r2s_prop[:, prop_idx],
                    )

In [ ]:
scaled_sim_echoes = np.full(sim_echoes.shape, np.nan)

plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}")
    plt.plot(sim_echoes[:,:, prop_idx])
    plt.ylabel("Raw Signal")
    plt.xlabel("Timepoints")
plt.show()

plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}")
    scaled_sim_echoes[:,:,prop_idx] = sim_echoes[:,:, prop_idx]/np.mean(sim_echoes[:,:, prop_idx], axis=0, keepdims=True)
    plt.plot(scaled_sim_echoes[:,:, prop_idx])
    plt.ylabel("Mean-Scaled Signal")
    plt.xlabel("Timepoints")
plt.show()


In [ ]:
plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}, mean scaled")
    plt.plot(tes, sim_echoes[:,:, prop_idx].T)
    plt.ylabel("Raw Signal")
    plt.xlabel("Echoe Time")
plt.show()

plt.figure(figsize=(15,15))
for prop_idx, prop in enumerate(proportion_s0_r2s):
    plt.subplot(3,2, prop_idx+1)
    plt.title(f"Proportion s0/r2s: {prop:.2f}, mean scaled")
    plt.plot(tes, scaled_sim_echoes[:,:, prop_idx].T)
    plt.ylabel("Mean-Scaled Signal")
    plt.xlabel("Echoe Time")
plt.show()